# Spot Overlay Viewer (Interactive)
Real-time spot overlay adjustment on MAR. Adjust sliders and the image updates automatically.

In [41]:
import numpy as np
import anndata
import os
import json
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import PatchCollection
from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display

In [42]:
def transform_spots(spots, shift_x, shift_y, flip_h, flip_v,
                    scale_x_log2, scale_y_log2, rotation_deg,
                    mode=None, scalefactor=1.0, display_scale=1.0):
    spots = spots.copy()
    if mode == 'export':
        shift_x = shift_x / scalefactor / display_scale
        shift_y = shift_y / scalefactor / display_scale
    spots[:, 0] += shift_x
    spots[:, 1] += shift_y
    if flip_h:
        spots[:, 0] = 2 * np.mean(spots[:, 0]) - spots[:, 0]
    if flip_v:
        spots[:, 1] = 2 * np.mean(spots[:, 1]) - spots[:, 1]
    cx, cy = np.mean(spots[:, 0]), np.mean(spots[:, 1])
    spots[:, 0] = cx + (spots[:, 0] - cx) * 2 ** scale_x_log2
    spots[:, 1] = cy + (spots[:, 1] - cy) * 2 ** scale_y_log2
    angle = np.deg2rad(rotation_deg)
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    centered = spots - [cx, cy]
    rotated = centered @ R.T
    return rotated + [cx, cy]

In [39]:
h5ad_path = widgets.Text(
    value='',
    placeholder='/path/to/file.h5ad',
    description='h5ad path:',
    layout=widgets.Layout(width='80%')
)
load_btn = widgets.Button(description='Load AnnData', button_style='success')
status_label = widgets.Label(value='No data loaded')

display(widgets.HBox([h5ad_path, load_btn, status_label]))

state = {}

def on_load(b):
    path = h5ad_path.value.strip()
    if not path or not os.path.exists(path):
        status_label.value = 'File not found!'
        return
    load_btn.disabled = True
    try:
        adata = anndata.read_h5ad(path)
        if len(adata.uns['spatial']) != 1:
            status_label.value = 'Error: multiple spatial entries'
            load_btn.disabled = False
            return
        lib_id = list(adata.uns['spatial'].keys())[0]
        img_data = adata.uns['spatial']
        image_info = img_data[lib_id]['images']['hires']
        if not isinstance(image_info, np.ndarray):
            status_label.value = 'No valid hires image'
            load_btn.disabled = False
            return
        image = Image.fromarray(image_info)
        max_size = 2000
        w, h = image.size
        scale = min(max_size / w, max_size / h, 1.0)
        image_resized = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
        spots = adata.obsm['spatial']
        valid_mask = ~np.isnan(spots).any(axis=1)
        scalefactor = img_data[lib_id]['scalefactors']['tissue_hires_scalef']
        display_scale = image_resized.size[0] / image.size[0]
        spots_scaled = spots[valid_mask] * scalefactor * display_scale
        spot_radius = img_data[lib_id]['scalefactors']['spot_diameter_fullres'] / 2
        state.update({
            'adata': adata, 'lib_id': lib_id,
            'image_original': image, 'image_resized': image_resized,
            'spots_full': spots, 'valid_mask': valid_mask,
            'spots_scaled': spots_scaled,
            'spots_scaled_orig': spots_scaled.copy(),
            'scalefactor': scalefactor, 'display_scale': display_scale,
            'spot_radius': spot_radius,
        })
        status_label.value = f'Loaded: {lib_id} — adjust sliders below'
        render()
    except Exception as e:
        status_label.value = f'Error: {e}'
    finally:
        load_btn.disabled = False

load_btn.on_click(on_load)

In [44]:
def linked_slider(desc, min_val, max_val, step, fmt='.5f'):
    slider = widgets.FloatSlider(value=0.0, min=min_val, max=max_val, step=step,
                                  description=desc, readout_format=fmt,
                                  style={'description_width': 'initial'},
                                  layout=widgets.Layout(width='100%'))
    entry = widgets.FloatText(value=0.0, step=step, layout=widgets.Layout(width='90px'))
    widgets.jslink((slider, 'value'), (entry, 'value'))
    return widgets.VBox([slider, entry]), slider

show_spots = widgets.Checkbox(value=True, description='Show Spots')
scalef_mult_box, scalef_mult = linked_slider('scalef mult', -5.0, 5.0, 0.01)
shift_x_box, shift_x_w = linked_slider('Shift X', -5000.0, 5000.0, 1.0)
shift_y_box, shift_y_w = linked_slider('Shift Y', -5000.0, 5000.0, 1.0)
zoom_box, zoom_w = linked_slider('Zoom', 0.1, 5.0, 0.1)

spot_diam_mult_box, spot_diam_mult = linked_slider('Spot diam mult', -5.0, 5.0, 0.01)
scale_x_box, scale_x_w = linked_slider('Scale X', -5.0, 5.0, 0.01)
scale_y_box, scale_y_w = linked_slider('Scale Y', -5.0, 5.0, 0.01)
rotation_box, rotation_w = linked_slider('Rotation (\u00b0)', -360.0, 360.0, 1.0)
flip_h_w = widgets.Checkbox(value=False, description='Flip Horiz')
flip_v_w = widgets.Checkbox(value=False, description='Flip Vert')

render_btn = widgets.Button(description='Render', button_style='info')
reset_btn = widgets.Button(description='Reset', button_style='warning')

advanced = widgets.Accordion(children=[widgets.VBox([
    spot_diam_mult_box, scale_x_box, scale_y_box, rotation_box,
    widgets.HBox([flip_h_w, flip_v_w])
])])
advanced.set_title(0, 'Advanced')
advanced.selected_index = None

controls = widgets.VBox([
    show_spots, scalef_mult_box, shift_x_box, shift_y_box, zoom_box,
    advanced, render_btn, reset_btn
], layout=widgets.Layout(width='30%'))

out_img = widgets.Image(format='jpeg')
img_box = widgets.Box([out_img], layout=widgets.Layout(
    width='70%', max_height='800px', overflow='scroll'))

display(widgets.HBox([controls, img_box]))

def on_reset(b):
    scalef_mult.value = 0.0
    shift_x_w.value = 0.0
    shift_y_w.value = 0.0
    spot_diam_mult.value = 0.0
    scale_x_w.value = 0.0
    scale_y_w.value = 0.0
    rotation_w.value = 0.0
    flip_h_w.value = False
    flip_v_w.value = False
    zoom_w.value = 1.0

def render(_=None):
    if 'adata' not in state:
        return
    try:
        zs = zoom_w.value
        img = state['image_resized'].copy()
        if zs != 1.0:
            w_img, h_img = img.size
            img = img.resize((int(w_img * zs), int(h_img * zs)), Image.LANCZOS)

        if show_spots.value:
            transformed = transform_spots(
                state['spots_scaled'], shift_x_w.value, shift_y_w.value,
                flip_h_w.value, flip_v_w.value,
                scale_x_w.value, scale_y_w.value, rotation_w.value,
            )
            transformed = transformed * 2 ** scalef_mult.value * zs
            r = (state['spot_radius']
                 * 2 ** spot_diam_mult.value
                 * state['scalefactor']
                 * 2 ** scalef_mult.value
                 * zs
                 * state['display_scale'])
            overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            for x, y in transformed:
                if not np.isnan(x) and not np.isnan(y):
                    draw.ellipse([x - r, y - r, x + r, y + r],
                                 fill=(0, 0, 255, 128), outline=(0, 0, 0, 200))
            img = img.convert('RGBA')
            img = Image.alpha_composite(img, overlay)

        buf = io.BytesIO()
        img.convert('RGB').save(buf, format='JPEG', quality=95)
        buf.seek(0)
        out_img.value = buf.getvalue()
    except Exception as e:
        print(f'Render error: {e}')

render_btn.on_click(lambda b: render())
reset_btn.on_click(on_reset)

for w in [show_spots, scalef_mult, shift_x_w, shift_y_w, zoom_w,
          spot_diam_mult, scale_x_w, scale_y_w, rotation_w, flip_h_w, flip_v_w]:
    w.observe(render, names='value')

render()

In [45]:
export_dir_w = widgets.Text(value='', placeholder='/path/to/export_dir', description='Export dir:', layout=widgets.Layout(width='80%'))
export_tsv_btn = widgets.Button(description='Export TSVs', button_style='primary')
export_h5ad_path = widgets.Text(value='', placeholder='/path/to/output.h5ad', description='H5AD path:', layout=widgets.Layout(width='80%'))
export_h5ad_btn = widgets.Button(description='Export H5AD', button_style='primary')
export_status = widgets.Label(value='')

display(widgets.VBox([
    widgets.HBox([export_dir_w, export_tsv_btn]),
    widgets.HBox([export_h5ad_path, export_h5ad_btn]),
    export_status
]))

def on_export_tsv(b):
    if 'adata' not in state:
        export_status.value = 'No data loaded'
        return
    out_dir = export_dir_w.value.strip()
    if not out_dir:
        export_status.value = 'Enter export directory'
        return
    os.makedirs(out_dir, exist_ok=True)
    adata = state['adata']
    spots_full = state['spots_full']
    valid_mask = state['valid_mask']
    scalefactor = state['scalefactor']
    spot_radius = state['spot_radius']
    transformed = transform_spots(
        spots_full[valid_mask], shift_x_w.value, shift_y_w.value,
        flip_h_w.value, flip_v_w.value,
        scale_x_w.value, scale_y_w.value, rotation_w.value,
        mode='export', scalefactor=scalefactor, display_scale=state['display_scale'],
    )
    transformed_coords = np.full_like(spots_full, np.nan)
    transformed_coords[valid_mask] = transformed
    barcodes = adata.obs_names
    tsv_path = os.path.join(out_dir, 'transformed_coords.tsv')
    with open(tsv_path, 'w') as f:
        f.write('barcode\tx\ty\n')
        for bc, coord in zip(barcodes, transformed_coords):
            x, y = coord
            x_str = '' if np.isnan(x) else f'{x:.2f}'
            y_str = '' if np.isnan(y) else f'{y:.2f}'
            f.write(f'{bc}\t{x_str}\t{y_str}\n')
    scalefactor_hires = scalefactor * 2 ** scalef_mult.value
    json_path = os.path.join(out_dir, 'scalefactors_json.json')
    with open(json_path, 'w') as jf:
        json.dump({
            'spot_diameter_fullres': float(2 * spot_radius * 2 ** spot_diam_mult.value),
            'tissue_hires_scalef': float(scalefactor_hires),
            'tissue_lowres_scalef': 1.0,
        }, jf, indent=2)
    export_status.value = f'Exported to: {out_dir}'

def on_export_h5ad(b):
    if 'adata' not in state:
        export_status.value = 'No data loaded'
        return
    save_path = export_h5ad_path.value.strip()
    if not save_path:
        export_status.value = 'Enter h5ad output path'
        return
    adata = state['adata'].copy()
    spots_full = state['spots_full']
    valid_mask = state['valid_mask']
    scalefactor = state['scalefactor']
    spot_radius = state['spot_radius']
    lib_id = state['lib_id']
    transformed = transform_spots(
        spots_full[valid_mask].astype(np.float64, copy=True),
        shift_x_w.value, shift_y_w.value,
        flip_h_w.value, flip_v_w.value,
        scale_x_w.value, scale_y_w.value, rotation_w.value,
        mode='export', scalefactor=scalefactor, display_scale=state['display_scale'],
    )
    transformed_coords = np.full_like(spots_full, np.nan, dtype=np.float64)
    transformed_coords[valid_mask] = transformed
    adata.obsm['spatial'] = transformed_coords
    scalefactor_hires = scalefactor * 2 ** scalef_mult.value
    adata.uns['spatial'][lib_id]['scalefactors'] = {
        'spot_diameter_fullres': float(2 * spot_radius * 2 ** spot_diam_mult.value),
        'tissue_hires_scalef': float(scalefactor_hires),
        'tissue_lowres_scalef': 1.0,
    }
    adata.write_h5ad(save_path)
    export_status.value = f'Saved: {save_path}'

export_tsv_btn.on_click(on_export_tsv)
export_h5ad_btn.on_click(on_export_h5ad)